# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [3]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [ ]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [6]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

assert index.ntotal == len(documents), "Jumlah vektor di index tidak sesuai dengan jumlah dokumen."

faiss.write_index(index, '../data/faq_index.faiss')

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [ ]:
def semantic_search(query, top_k=3):
    if not 1 <= top_k <= index.ntotal:
        raise ValueError(f"Parameter top_k harus berada di antara 1 dan {index.ntotal}")

    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.61730665)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.6187018)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.634567)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [15]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )

    prompt = f"""Persona:
Kamu adalah asisten customer service e-commerce yang ramah, profesional, dan membantu.

Task:
Jawab pertanyaan pengguna hanya berdasarkan informasi dalam konteks FAQ yang diberikan.
Jangan menambahkan informasi, kebijakan, atau asumsi yang tidak didukung oleh konteks.
Jika konteks tidak memuat informasi yang cukup untuk menjawab pertanyaan, sampaikan bahwa informasi tersebut belum tersedia dan sarankan pengguna menghubungi customer service.

Context:
{context}

Pertanyaan Pengguna:
{query}

Format:
Berikan jawaban dalam bahasa Indonesia yang jelas, singkat, dan natural.
Jawab langsung pertanyaan pengguna tanpa menyebutkan proses semantic search atau retrieval.

Jawaban:"""

    return prompt

## 7. Mengintegrasikan LLM

In [17]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.8-flash',  
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo! Ya, Anda bisa membayar dengan cicilan. Fitur cicilan saat ini tersedia untuk transaksi menggunakan kartu kredit tertentu dengan nominal minimal yang ditentukan oleh pihak bank. 

Ada hal lain yang bisa kami bantu?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [19]:
test_queries = [
    "Bagaimana cara melakukan pemesanan di platform ini?",
    "Apakah saya bisa memesan tanpa membuat akun?",
    "Paket saya kok belum nyampe padahal sudah lewat perkiraan?",
    "Gimana caranya supaya pesanan yang sudah dibuat dibatalkan?",
    "Apakah perusahaan menyediakan layanan penyewaan mobil?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
for source in sources:
    print(
        f"Rank {source['rank']} | "
        f"Distance: {source['distance']:.4f} | "
        f"FAQ: {source['question']}"
    )
    print('---')

Query: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Halo! Untuk melakukan pemesanan di platform kami, Anda cukup memilih produk yang diinginkan, klik **'Tambah ke Keranjang'**, lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan memilih metode pembayaran.
Query: Apakah saya bisa memesan tanpa membuat akun?
Jawaban: Halo! Saat ini pemesanan hanya dapat dilakukan setelah Anda membuat akun, ya. Hal ini bertujuan agar Anda dapat melacak status pesanan dengan mudah.
Query: Paket saya kok belum nyampe padahal sudah lewat perkiraan?
Jawaban: Halo! Jika paket Anda sudah melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service kami dengan menyertakan nomor resi agar dapat ditelusuri lebih lanjut.
Query: Gimana caranya supaya pesanan yang sudah dibuat dibatalkan?
Jawaban: Halo! Anda dapat membatalkan pesanan melalui halaman **'Pesanan Saya'** selama status pesanan masih **'Diproses'**. 

Perlu diperhatikan bahwa setelah status pesanan be

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._